# Week 49, Day 3: Timed set 1

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w049d3.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


*Interactive figure on the website: The timer keeps running if you reload. It doesn't stop you working when time is up..*

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: P1: Merging intervals

Write `merge_intervals(intervals)`: given `(start, end)` pairs (start ≤ end, any order), merge every pair that overlaps **or touches** (`(1, 3)` and `(3, 5)` merge), returning the merged intervals sorted by start. Then `total_covered(intervals)`: the total length covered by at least one interval.

In [ ]:
def merge_intervals(intervals):
    ___

def total_covered(intervals):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef test_merge():\n    \"merging\"\n    assert merge_intervals([(1, 3), (8, 10), (2, 6), (15, 18)]) == [(1, 6), (8, 10), (15, 18)]\n    assert merge_intervals([(1, 3), (3, 5)]) == [(1, 5)] and merge_intervals([]) == [] and merge_intervals([(4, 4)]) == [(4, 4)]\n    assert total_covered([(1, 3), (2, 6), (8, 10)]) == 7\n    rng = random.Random(1)\n    for _ in range(200):\n        ivs = [(s, s + rng.randint(0, 5)) for s in (rng.randint(0, 30) for _ in range(rng.randint(0, 8)))]\n        m = merge_intervals(ivs)\n        assert all(a[1] < b[0] for a, b in zip(m, m[1:]))\n        covered = {x + 0.5 for s, e in ivs for x in range(s, e)}\n        assert total_covered(ivs) == len(covered)\n        assert all(any(s <= a and b <= e for s, e in m) for a, b in ivs)")

### Exercise: P2: A stack that knows its minimum

Write `MinStack` with `push(x)`, `pop()` (returning the value; `IndexError` when empty), `top()`, `get_min()` (`IndexError` when empty) and `len`, where **every** operation is O(1). Keep a second stack of the minimum so far alongside the main one.

In [ ]:
class MinStack:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random, time\ndef test_minstack():\n    \"minimums\"\n    s = MinStack()\n    for x in [5, 3, 7, 3, 1]:\n        s.push(x)\n    assert s.get_min() == 1 and s.pop() == 1 and s.get_min() == 3 and s.pop() == 3 and s.get_min() == 3\n    assert s.top() == 7 and len(s) == 3\n    for f in (MinStack().pop, MinStack().get_min, MinStack().top):\n        try:\n            f()\n            assert False\n        except IndexError:\n            pass\n    rng = random.Random(2)\n    s, ref = MinStack(), []\n    for _ in range(3000):\n        if ref and rng.random() < 0.4:\n            assert s.pop() == ref.pop()\n        else:\n            x = rng.randint(-50, 50)\n            s.push(x)\n            ref.append(x)\n        if ref:\n            assert s.get_min() == min(ref)\n    big = MinStack()\n    for x in range(100_000):\n        big.push(-x)\n    t0 = time.perf_counter()\n    for _ in range(100_000):\n        big.get_min()\n    assert time.perf_counter() - t0 < 1")

### Exercise: P3: Copying a list with extra links

Each `Node` has `value`, `next`, and `other`, which points to any node in the same list or is `None`. Write `deep_copy(head)`: a completely new list with the same values, where each copy's `next` and `other` point to the corresponding **copies**. Use a dict from original node to its copy, in two passes. O(n).

In [ ]:
class Node:
    def __init__(self, value, next=None, other=None):
        self.value, self.next, self.other = value, next, other

def deep_copy(head):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef make(values, links):\n    nodes = [Node(v) for v in values]\n    for a, b in zip(nodes, nodes[1:]):\n        a.next = b\n    for i, j in enumerate(links):\n        nodes[i].other = nodes[j] if j is not None else None\n    return nodes\ndef shape(head):\n    nodes, n = [], head\n    while n:\n        nodes.append(n)\n        n = n.next\n    index = {id(x): i for i, x in enumerate(nodes)}\n    return [(x.value, index[id(x.other)] if x.other else None) for x in nodes], {id(x) for x in nodes}\ndef test_copy():\n    \"copies\"\n    assert deep_copy(None) is None\n    rng = random.Random(3)\n    for _ in range(100):\n        n = rng.randint(1, 10)\n        nodes = make([rng.randint(0, 5) for _ in range(n)], [rng.choice([None] + list(range(n))) for _ in range(n)])\n        before, ids = shape(nodes[0])\n        c = deep_copy(nodes[0])\n        after, new_ids = shape(c)\n        assert after == before and not (ids & new_ids), \"new nodes, same shape\"\n        assert shape(nodes[0])[0] == before, \"leave the original alone\"")

### Exercise: P4: The kth smallest in a sorted grid

Each row and each column of a square grid of numbers is sorted in increasing order. Write `kth_smallest(grid, k)` (k from 1): the kth smallest value overall, in O(k log n), with a heap holding one candidate per row (start with each row's first item; after popping from a row, push that row's next item). `IndexError` if k is out of range. Then `count_at_most(grid, x)`: how many values are ≤ x, in O(n) by walking a staircase from the bottom-left corner.

In [ ]:
import heapq

def kth_smallest(grid, k):
    ___

def count_at_most(grid, x):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef test_grid():\n    \"grids\"\n    g = [[1, 5, 9], [10, 11, 13], [12, 13, 15]]\n    assert kth_smallest(g, 8) == 13 and kth_smallest(g, 1) == 1 and kth_smallest(g, 9) == 15\n    assert count_at_most(g, 12) == 6 and count_at_most(g, 0) == 0 and count_at_most(g, 99) == 9\n    for k in (0, 10):\n        try:\n            kth_smallest(g, k)\n            assert False\n        except IndexError:\n            pass\n    rng = random.Random(4)\n    for _ in range(100):\n        n = rng.randint(1, 7)\n        grid = [[0] * n for _ in range(n)]\n        for i in range(n):\n            for j in range(n):\n                grid[i][j] = max(grid[i - 1][j] if i else 0, grid[i][j - 1] if j else 0) + rng.randint(0, 3)\n        flat = sorted(v for row in grid for v in row)\n        k = rng.randint(1, n * n)\n        assert kth_smallest(grid, k) == flat[k - 1]\n        x = rng.randint(0, 30)\n        assert count_at_most(grid, x) == sum(1 for v in flat if v <= x)")

### Exercise: P5: A word ladder

A word ladder changes one letter at a time, every step being a word from a given list. Write `ladder_length(start, end, words)`: the number of words in the shortest ladder from `start` to `end` (counting both), or 0 if there's none (`end` must be in `words`; `start` needn't be). Use a breadth-first search with a `deque`. To find neighbours quickly, group words by **patterns** with one letter replaced by `*` (so "hot" is in `*ot`, `h*t` and `ho*`) in a dict built once. Then `ladder(start, end, words)`: one shortest ladder as a list (any one), or `[]`, by remembering each word's predecessor.

In [ ]:
from collections import deque

def ladder_length(start, end, words):
    ___

def ladder(start, end, words):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random, string\ndef test_ladder():\n    \"ladders\"\n    words = [\"hot\", \"dot\", \"dog\", \"lot\", \"log\", \"cog\"]\n    assert ladder_length(\"hit\", \"cog\", words) == 5 and ladder_length(\"hit\", \"cog\", words[:-1]) == 0\n    path = ladder(\"hit\", \"cog\", words)\n    assert len(path) == 5 and path[0] == \"hit\" and path[-1] == \"cog\" and all(w in words for w in path[1:])\n    assert all(sum(a != b for a, b in zip(x, y)) == 1 for x, y in zip(path, path[1:]))\n    assert ladder_length(\"aa\", \"aa\", [\"aa\"]) == 1 and ladder(\"ab\", \"cd\", [\"cd\"]) == []\ndef test_random():\n    \"against a slow search\"\n    rng = random.Random(5)\n    for _ in range(60):\n        words = list({\"\".join(rng.choice(\"abc\") for _ in range(3)) for _ in range(rng.randint(1, 20))})\n        start = \"\".join(rng.choice(\"abc\") for _ in range(3))\n        end = rng.choice(words)\n        dist, frontier, seen = 1, [start], {start}\n        want = 0\n        while frontier:\n            if end in frontier:\n                want = dist\n                break\n            nxt = []\n            for w in frontier:\n                for v in words:\n                    if v not in seen and sum(a != b for a, b in zip(v, w)) == 1:\n                        seen.add(v)\n                        nxt.append(v)\n            frontier, dist = nxt, dist + 1\n        assert ladder_length(start, end, words) == want\n        assert len(ladder(start, end, words)) == want")

## How to use this

Treat the next 100 minutes as the real thing: notes closed, no tutor, no web, timer on. Run your code and press **Check** as often as you like. Each problem has a hint and a model solution: **don't open them until the timer has finished**. Then go back through every problem, open the solutions, and compare.

Before writing code for each problem, write a comment at the top of your answer: the pattern you'll use and its time cost. It takes thirty seconds and saves many minutes of coding the wrong idea.

**The marks.**

| Part | What | Marks |
|---|---|---|
| A | 10 quiz questions | 10 (1 each) |
| B | 5 problems | 80 |
| | **Total** | **90** |

Count full marks for a problem only when every check passes. 54 (60%) is a good result; 72 (80%) an excellent one.

## Part A: quiz

## Part B: five problems

## Afterwards

When the timer ends, add up your marks, then go back through each problem with its hint and solution. For every mark you dropped, decide which kind of reason it was:

1. **Didn't know it**: revise the topic from the review days (and the week they point to).
2. **Picked the wrong pattern**: you knew the tools but reached for the wrong one. Write down what in the problem should have pointed to the right one.
3. **Edge case**: the main case worked; an empty input, a tie or a boundary didn't. Tomorrow, test small and extreme cases before pressing Check.
4. **Ran out of time**: do the problems you're sure of first, and move on after 15 minutes stuck.

Which patterns did today's problems need?

- P1: sort, then a single sweep (Week 19 and Week 41).
- P2: a second stack of running minimums, the same "store the answer so far alongside" idea as prefix sums.
- P3: a dict from old nodes to new ones (Week 44), in two passes so every copy exists before it's linked.
- P4: a heap of one frontier item per row (Week 47 Day 5), and a staircase walk that uses both sort orders.
- P5: breadth-first search with a queue (Week 43), plus a dict of patterns so neighbours are found without comparing every pair of words. Graphs, next week's topic in all but name, are mostly this.

Tomorrow: timed set 2, with no hints and no solutions until Saturday's debrief.